# Notebook 08 — Error Analysis

> **阶段**：Stage 4 Evaluate · **预计时间**：30–50 分钟（CPU 可完成） · **平台**：Kaggle Notebook

目标是回答「模型到底不会什么」，而不是「模型分数是多少」。


# Learning Objectives

- 建立可复查的错误分类学（Error Taxonomy）；
- 从预测与 GT 自动生成 error_cases.json（含证据路径）；
- 选择 Worst / Best-Improvement / Regression 三类案例；
- 用错误分布提出下一步的研究假设。


# Why This Matters

同一 Overall 分数背后可以是完全不同的失败模式：OCR 错字、漏内容、表格丢失还是幻觉。错误分析把「分数差」变成「可研究的问题」，是论文问题形成前的最后一块拼图。


# Concepts

错误分类学（教学启发式，需人工复核）：

```text
OCR Error / Layout Error / Reading Order Error
Table Error / Formula Error / Missing Content
Hallucination / Repetition / Structure Error
```

自动分类依据：归一化编辑距离、SequenceMatcher 的缺失/插入占比、重复片段检测、页面是否含表格/公式而预测缺失对应标记。所有案例带 evidence 路径，便于逐条人工复查。


## Step 1 — 生成错误案例库


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data, error_analysis
from src.config import load_config, project_root

cfg = load_config()
data_root = data.find_dataset_root()
annotations = data.load_annotations(data_root)
pred_dir = project_root() / cfg['paths']['baseline_dir'] / 'predictions'

cases = error_analysis.build_error_cases(
    pred_dir, annotations, data_root,
    output_path=project_root() / 'results' / 'error_cases.json',
)
print('案例数:', len(cases))
print('错误类型分布:', error_analysis.taxonomy_summary(cases))


## Step 2 — 最差案例 Top N


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import json
from src import error_analysis

worst = error_analysis.select_worst_cases(cases, n=10)
for c in worst[:5]:
    print(json.dumps(c, ensure_ascii=False))


## Step 3 — 改善最大与退化案例（训练前后对比）

需要两份预测：baseline 与 fine-tuned。没有 SFT 预测时，先用两个 Prompt 目录演示同一对比函数（变量不同但流程相同）。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data, error_analysis
from src.config import project_root

base_dir = project_root() / 'results' / 'prompt_v0' / 'predictions'
other_dir = project_root() / 'results' / 'prompt_v1' / 'predictions'
if base_dir.is_dir() and other_dir.is_dir():
    base_cases = error_analysis.build_error_cases(base_dir, annotations, data_root)
    other_cases = error_analysis.build_error_cases(other_dir, annotations, data_root)
    comp = error_analysis.select_improvement_cases(base_cases, other_cases, n=5)
    print('improved:', [c['image_id'] for c in comp['improved']])
    print('regressed:', [c['image_id'] for c in comp['regressed']])
else:
    print('先运行 Notebook 03 生成 prompt_v0/v1 预测，或改用 SFT 预测目录。')


## Step 4 — 复核一个最差案例

自动分类只是入口。挑 1 个最差案例，打开 evidence 里的 prediction 与对应 GT 页面，人工判断：错误来自模型、转换还是评测对齐？把结论写回该案例的 notes 字段。


# What You Should Observe

- 错误类型分布通常不均：某 1–2 类占主导 → 这就是研究切入点；
- worst 案例常有共同特征（多栏、手写、低分辨率、表格复杂）；
- 改善与退化并存的页面说明训练不是单向变好。


# Research Checkpoint

> **模型到底不会什么？** 用你的 taxonomy 统计和 2 个具体案例回答，并说明这个结论与「平均分」的关系。

**TODO：** 答案写入 `results/nb08/research_checkpoint.md`。


# Exercises

1. **TODO：** 对 Top5 最差案例逐条人工复核，把「模型错」与「评测/转换错」分开，更新 notes；
2. **TODO：** 按 document_type 分组统计错误类型，找出系统性弱点子群；
3. **TODO：** 写一个新的启发式分类器（例如「表格行列数不匹配」），加进 classify_case 并评估其对错率。


# Takeaways

- 分数说「差多少」，错误分析说「差在哪」；
- 启发式分类需要证据路径与人工复核，不能直接当作结论；
- 错误分布是研究假设的原材料。

**下一步**：[Notebook 09](09_Ablation_Study.ipynb) — 用受控实验验证变量。
